In [12]:
import pandas as pd
import numpy as np

In [14]:
df=pd.read_csv("Employee data.csv")

In [15]:
# before cleaning, saving counts for summary table
before_rows=df.shape[0]
before_missing=int(df.isna().sum().sum())
before_dup=int(df.duplicated().sum())

print("before rows:",before_rows)
print("before missing:",before_missing)
print("before duplicates:",before_dup)
print(df.head())

before rows: 474
before missing: 1
before duplicates: 0
    id  gender       bdate  educ    jobcat   salary  salbegin  jobtime  \
0  1.0    Male  1952-02-03    15   Manager  57000.0   27000.0     98.0   
1  2.0    Male  1958-05-23    16  Clerical  40200.0   18750.0     98.0   
2  3.0  Female  1929-07-26    12  Clerical  21450.0   12000.0     98.0   
3  4.0  Female  1947-04-15     8  Clerical  21900.0   13200.0     98.0   
4  5.0    Male  1955-02-09    15  Clerical  45000.0   21000.0     98.0   

  prevexp minority  
0   144.0       No  
1    36.0       No  
2   381.0       No  
3   190.0       No  
4   138.0       No  


In [ ]:
# cleaning
# In this data missing is written as text 'missing', not NaN
# so first i am changing it to NaN
df=df.replace("missing",np.nan)
df=df.replace("Missing",np.nan)

# some numbers are stored as text, changing to number
df["salary"]=pd.to_numeric(df["salary"],errors="coerce")
df["salbegin"]=pd.to_numeric(df["salbegin"],errors="coerce")
df["educ"]=pd.to_numeric(df["educ"],errors="coerce")
df["jobtime"]=pd.to_numeric(df["jobtime"],errors="coerce")
df["prevexp"]=pd.to_numeric(df["prevexp"],errors="coerce")


df["bdate"]=pd.to_datetime(df["bdate"],errors="coerce")
df["id"]=pd.to_numeric(df["id"],errors="coerce").astype("Int64")



In [27]:
# Handle missing values
# this all are in numbers,used median
df["salary"]=df["salary"].fillna(df["salary"].median())
df["salbegin"]=df["salbegin"].fillna(df["salbegin"].median())
df["educ"]=df["educ"].fillna(df["educ"].median())

df["jobtime"]=df["jobtime"].fillna(df["jobtime"].median())
df["prevexp"]=df["prevexp"].fillna(df["prevexp"].median())

# jobcat is category so used mode
df["jobcat"]=df["jobcat"].fillna(df["jobcat"].mode()[0])
df["gender"]=df["gender"].fillna(df["gender"].mode()[0])
df["minority"]=df["minority"].fillna(df["minority"].mode()[0])

df["bdate"]=df["bdate"].fillna(df["bdate"].mode()[0])


print(df.isna().sum())


id          0
gender      0
bdate       0
educ        0
jobcat      0
salary      0
salbegin    0
jobtime     0
prevexp     0
minority    0
dtype: int64


In [19]:
# cleaning text and removing extra spaces
df["gender"]=df["gender"].astype(str).str.strip().str.title()
df["jobcat"]=df["jobcat"].astype(str).str.strip().str.title()
df["minority"]=df["minority"].astype(str).str.strip().str.title()


In [20]:
df=df.drop_duplicates()
print("duplicates after:",df.duplicated().sum())

duplicates after: 0


In [28]:
# salary outlier using IQR
Q1=df["salary"].quantile(0.25)
Q3=df["salary"].quantile(0.75)
IQR=Q3-Q1
upper=Q3+1.5*IQR
print("upper:", upper)

upper: 56343.75


In [22]:
# i did not delete outlier, maybe it is real salary
# so i capped it to upper value
df["salary"]=df["salary"].clip(upper=upper)

In [29]:
# after cleaning counts
after_rows=df.shape[0]
after_missing=int(df.isna().sum().sum())
after_dup=int(df.duplicated().sum())

In [30]:
# before after summary table
summary=pd.DataFrame({"Stage":["Before","After"],
                      "Rows":[before_rows,after_rows],
                      "Missing values":[before_missing,after_missing],
                      "Duplicates":[before_dup,after_dup]})

print(summary)

    Stage  Rows  Missing values  Duplicates
0  Before   474               1           0
1   After   474               0           0


In [31]:
df.to_csv("clean_data.csv",index=False)
print("clean_data.csv saved")
print(df.head())

clean_data.csv saved
   id  gender      bdate  educ    jobcat    salary  salbegin  jobtime  \
0   1    Male 1952-02-03    15   Manager  56343.75   27000.0     98.0   
1   2    Male 1958-05-23    16  Clerical  40200.00   18750.0     98.0   
2   3  Female 1929-07-26    12  Clerical  21450.00   12000.0     98.0   
3   4  Female 1947-04-15     8  Clerical  21900.00   13200.0     98.0   
4   5    Male 1955-02-09    15  Clerical  45000.00   21000.0     98.0   

   prevexp minority  
0    144.0       No  
1     36.0       No  
2    381.0       No  
3    190.0       No  
4    138.0       No  
